> Note: This notebook is an exercise workbook companion. Some cells intentionally contain `TODO` prompts or `NotImplementedError` placeholders for the reader to complete. For fully maintained runnable reference code, see `src/`, `tests/`, and the printed listings in the book.

# Chapter 1: Convex Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/blob/main/notebooks/ch01_convex_optimization.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:
1. Understand and verify convexity of functions and sets
2. Visualize convex sets (balls, polyhedra, cones)
3. Implement gradient descent for convex and non-convex functions
4. Understand and verify KKT optimality conditions
5. Use interactive widgets to explore optimization parameters

---


In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib.patches import Polygon, Circle, FancyArrowPatch
from matplotlib.collections import PatchCollection
import warnings
warnings.filterwarnings('ignore')

# Try to import ipywidgets (works in Jupyter/Colab)
try:
    from ipywidgets import interact, interactive, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets not available. Install with: pip install ipywidgets")

# For Colab, install ipywidgets if needed
# !pip install ipywidgets

np.random.seed(42)
print("Setup complete!")


## 1. Interactive Convexity Checker

A function $f: \mathbb{R}^n \to \mathbb{R}$ is **convex** if for all $x, y$ in its domain and all $\lambda \in [0, 1]$:

$$f(\lambda x + (1-\lambda) y) \leq \lambda f(x) + (1-\lambda) f(y)$$

This means the line segment between any two points on the graph lies above or on the graph itself.


In [ ]:
def check_convexity_1d(f, x_range=(-5, 5), n_tests=1000, tol=1e-6):
    """
    Check if a 1D function is convex by sampling.

    Parameters:
    -----------
    f : callable
        Function to test
    x_range : tuple
        Range to test over
    n_tests : int
        Number of random tests
    tol : float
        Tolerance for numerical errors

    Returns:
    --------
    is_convex : bool
    violations : list of tuples (x1, x2, lambda) where convexity is violated
    """
    violations = []

    for _ in range(n_tests):
        x1 = np.random.uniform(x_range[0], x_range[1])
        x2 = np.random.uniform(x_range[0], x_range[1])
        lam = np.random.uniform(0, 1)

        # Convexity condition: f(lam*x1 + (1-lam)*x2) <= lam*f(x1) + (1-lam)*f(x2)
        x_mid = lam * x1 + (1 - lam) * x2
        lhs = f(x_mid)
        rhs = lam * f(x1) + (1 - lam) * f(x2)

        if lhs > rhs + tol:
            violations.append((x1, x2, lam, lhs - rhs))

    return len(violations) == 0, violations

def visualize_convexity_check(f, f_name, x_range=(-3, 3)):
    """Visualize convexity check for a function."""
    x = np.linspace(x_range[0], x_range[1], 500)
    y = f(x)

    is_convex, violations = check_convexity_1d(f, x_range)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Function with convexity illustration
    ax1 = axes[0]
    ax1.plot(x, y, 'b-', linewidth=2, label=f_name)

    # Pick two points and show the line between them
    x1, x2 = -2, 2
    y1, y2 = f(x1), f(x2)
    ax1.plot([x1, x2], [y1, y2], 'r--', linewidth=2, label='Line segment')
    ax1.scatter([x1, x2], [y1, y2], c='red', s=100, zorder=5)

    # Show a midpoint
    lam = 0.5
    x_mid = lam * x1 + (1 - lam) * x2
    y_mid_line = lam * y1 + (1 - lam) * y2
    y_mid_func = f(x_mid)

    ax1.scatter([x_mid], [y_mid_line], c='green', s=100, marker='^',
                label=f'Line at midpoint: {y_mid_line:.2f}', zorder=5)
    ax1.scatter([x_mid], [y_mid_func], c='purple', s=100, marker='v',
                label=f'Function at midpoint: {y_mid_func:.2f}', zorder=5)

    ax1.axvline(x_mid, color='gray', linestyle=':', alpha=0.5)

    status = "CONVEX" if is_convex else "NOT CONVEX"
    ax1.set_title(f'{f_name}: {status}', fontsize=14)
    ax1.set_xlabel('x')
    ax1.set_ylabel('f(x)')
    ax1.legend(loc='upper center')
    ax1.grid(True, alpha=0.3)

    # Plot 2: Multiple lambda values
    ax2 = axes[1]
    ax2.plot(x, y, 'b-', linewidth=2)

    lambdas = [0.2, 0.4, 0.6, 0.8]
    colors = plt.cm.viridis(np.linspace(0, 1, len(lambdas)))

    for lam, c in zip(lambdas, colors):
        x_mid = lam * x1 + (1 - lam) * x2
        y_line = lam * y1 + (1 - lam) * y2
        y_func = f(x_mid)

        ax2.scatter([x_mid], [y_line], c=[c], s=80, marker='^', edgecolors='black')
        ax2.scatter([x_mid], [y_func], c=[c], s=80, marker='v', edgecolors='black')
        ax2.plot([x_mid, x_mid], [y_func, y_line], c=c, linestyle='-', alpha=0.7)

    ax2.plot([x1, x2], [y1, y2], 'r--', linewidth=2)
    ax2.scatter([x1, x2], [y1, y2], c='red', s=100, zorder=5)
    ax2.set_title('Convexity at multiple points\n(^ = line, v = function)', fontsize=14)
    ax2.set_xlabel('x')
    ax2.set_ylabel('f(x)')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return is_convex

# Test various functions
print("=" * 60)
print("CONVEXITY CHECKER RESULTS")
print("=" * 60)

# Convex functions
convex_funcs = [
    (lambda x: x**2, "f(x) = x^2"),
    (lambda x: np.exp(x), "f(x) = e^x"),
    (lambda x: np.abs(x), "f(x) = |x|"),
    (lambda x: x**4, "f(x) = x^4"),
]

# Non-convex functions
nonconvex_funcs = [
    (lambda x: np.sin(x), "f(x) = sin(x)"),
    (lambda x: -x**2, "f(x) = -x^2"),
    (lambda x: x**3, "f(x) = x^3"),
    (lambda x: np.cos(x), "f(x) = cos(x)"),
]

for f, name in convex_funcs:
    visualize_convexity_check(f, name)


## 2. Visualizing Convex Sets

A set $C$ is **convex** if for any two points $x, y \in C$, the entire line segment connecting them is also in $C$:

$$\lambda x + (1-\lambda) y \in C \quad \forall \lambda \in [0, 1]$$

Common convex sets include:
- **Balls**: $\{x : \|x - c\| \leq r\}$
- **Polyhedra**: $\{x : Ax \leq b\}$
- **Cones**: $\{x : x = \sum_i \alpha_i v_i, \alpha_i \geq 0\}$


In [ ]:
def visualize_convex_sets():
    """Visualize different types of convex sets."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # 1. Ball (L2)
    ax = axes[0, 0]
    theta = np.linspace(0, 2*np.pi, 100)
    r = 2
    x_circle = r * np.cos(theta)
    y_circle = r * np.sin(theta)
    ax.fill(x_circle, y_circle, alpha=0.3, color='blue')
    ax.plot(x_circle, y_circle, 'b-', linewidth=2)

    # Show convexity with two points
    p1, p2 = np.array([-1.5, 0.5]), np.array([1, 1.2])
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r-', linewidth=2)
    ax.scatter([p1[0], p2[0]], [p1[1], p2[1]], c='red', s=100, zorder=5)
    ax.set_title('L2 Ball (Euclidean)', fontsize=12)
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    # 2. Ball (L1) - Diamond
    ax = axes[0, 1]
    r = 2
    diamond = np.array([[r, 0], [0, r], [-r, 0], [0, -r], [r, 0]])
    ax.fill(diamond[:, 0], diamond[:, 1], alpha=0.3, color='green')
    ax.plot(diamond[:, 0], diamond[:, 1], 'g-', linewidth=2)

    p1, p2 = np.array([-0.5, 0.8]), np.array([0.8, -0.3])
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r-', linewidth=2)
    ax.scatter([p1[0], p2[0]], [p1[1], p2[1]], c='red', s=100, zorder=5)
    ax.set_title('L1 Ball (Manhattan)', fontsize=12)
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    # 3. Ball (L-infinity) - Square
    ax = axes[0, 2]
    r = 2
    square = np.array([[-r, -r], [r, -r], [r, r], [-r, r], [-r, -r]])
    ax.fill(square[:, 0], square[:, 1], alpha=0.3, color='orange')
    ax.plot(square[:, 0], square[:, 1], color='orange', linewidth=2)

    p1, p2 = np.array([-1.5, 1.5]), np.array([1.5, -1])
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r-', linewidth=2)
    ax.scatter([p1[0], p2[0]], [p1[1], p2[1]], c='red', s=100, zorder=5)
    ax.set_title('L-infinity Ball (Chebyshev)', fontsize=12)
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    # 4. Polyhedron (general)
    ax = axes[1, 0]
    vertices = np.array([[0, 0], [3, 0], [4, 2], [2, 4], [-1, 3], [0, 0]])
    ax.fill(vertices[:, 0], vertices[:, 1], alpha=0.3, color='purple')
    ax.plot(vertices[:, 0], vertices[:, 1], color='purple', linewidth=2)

    p1, p2 = np.array([0.5, 1]), np.array([3, 2.5])
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r-', linewidth=2)
    ax.scatter([p1[0], p2[0]], [p1[1], p2[1]], c='red', s=100, zorder=5)
    ax.set_title('Polyhedron', fontsize=12)
    ax.set_xlim(-2, 5)
    ax.set_ylim(-1, 5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    # 5. Cone
    ax = axes[1, 1]
    cone_verts = np.array([[0, 0], [4, 1], [4, 4], [0, 0]])
    ax.fill(cone_verts[:, 0], cone_verts[:, 1], alpha=0.3, color='cyan')
    ax.plot(cone_verts[:, 0], cone_verts[:, 1], color='cyan', linewidth=2)

    # Show rays from origin
    ax.arrow(0, 0, 3.5, 0.875, head_width=0.15, color='cyan', linewidth=1.5)
    ax.arrow(0, 0, 2.5, 2.5, head_width=0.15, color='cyan', linewidth=1.5)

    p1, p2 = np.array([1, 0.5]), np.array([2.5, 2])
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r-', linewidth=2)
    ax.scatter([p1[0], p2[0]], [p1[1], p2[1]], c='red', s=100, zorder=5)
    ax.set_title('Cone', fontsize=12)
    ax.set_xlim(-0.5, 5)
    ax.set_ylim(-0.5, 5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    # 6. Non-convex set (for comparison)
    ax = axes[1, 2]
    theta = np.linspace(0, 2*np.pi, 100)
    # Star shape (non-convex)
    r_star = 2 + np.cos(5*theta)
    x_star = r_star * np.cos(theta)
    y_star = r_star * np.sin(theta)
    ax.fill(x_star, y_star, alpha=0.3, color='red')
    ax.plot(x_star, y_star, 'r-', linewidth=2)

    # Show line segment leaving the set
    p1, p2 = np.array([-2, 1]), np.array([2, 1])
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'k-', linewidth=2)
    ax.scatter([p1[0], p2[0]], [p1[1], p2[1]], c='black', s=100, zorder=5)
    ax.scatter([0], [1], c='yellow', s=100, marker='x', linewidth=3, zorder=5)
    ax.annotate('Outside!', xy=(0, 1), xytext=(0.5, 2), fontsize=10,
                arrowprops=dict(arrowstyle='->', color='black'))
    ax.set_title('NON-CONVEX Set (Star)', fontsize=12)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    plt.suptitle('Convex Sets Visualization\n(Red lines show line segments between points)',
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

visualize_convex_sets()


## 3. Gradient Descent: Convex vs Non-Convex

Gradient descent update rule:
$$x_{k+1} = x_k - \alpha \nabla f(x_k)$$

For **convex** functions:
- Any local minimum is a global minimum
- Gradient descent converges to the global optimum

For **non-convex** functions:
- May have multiple local minima
- Gradient descent may get stuck in local minima


In [ ]:
def gradient_descent_1d(f, grad_f, x0, lr=0.1, max_iter=100, tol=1e-8):
    """
    1D Gradient descent with history tracking.

    Parameters:
    -----------
    f : callable
        Objective function
    grad_f : callable
        Gradient of objective function
    x0 : float
        Initial point
    lr : float
        Learning rate
    max_iter : int
        Maximum iterations
    tol : float
        Convergence tolerance

    Returns:
    --------
    x_history : list
        History of x values
    f_history : list
        History of function values
    """
    x = x0
    x_history = [x]
    f_history = [f(x)]

    for i in range(max_iter):
        grad = grad_f(x)
        x_new = x - lr * grad

        x_history.append(x_new)
        f_history.append(f(x_new))

        if abs(x_new - x) < tol:
            break
        x = x_new

    return x_history, f_history

def compare_convex_nonconvex():
    """Compare GD behavior on convex vs non-convex functions."""

    # Convex function: f(x) = x^2
    f_convex = lambda x: x**2
    grad_convex = lambda x: 2*x

    # Non-convex function: f(x) = x^4 - 2x^2 + 0.5x (has multiple local minima)
    f_nonconvex = lambda x: x**4 - 2*x**2 + 0.5*x
    grad_nonconvex = lambda x: 4*x**3 - 4*x + 0.5

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Convex function plot
    x = np.linspace(-3, 3, 500)
    y_convex = f_convex(x)

    ax = axes[0, 0]
    ax.plot(x, y_convex, 'b-', linewidth=2, label='f(x) = x^2')

    # Run GD from different starting points
    starting_points = [-2.5, 2.5, 1.0]
    colors = ['red', 'green', 'orange']

    for x0, c in zip(starting_points, colors):
        x_hist, f_hist = gradient_descent_1d(f_convex, grad_convex, x0, lr=0.2)
        ax.plot(x_hist, f_hist, 'o-', color=c, markersize=6,
                label=f'Start x0={x0}', alpha=0.7)
        ax.scatter([x_hist[0]], [f_hist[0]], color=c, s=150, marker='s',
                   edgecolors='black', zorder=5)
        ax.scatter([x_hist[-1]], [f_hist[-1]], color=c, s=150, marker='*',
                   edgecolors='black', zorder=5)

    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.set_title('CONVEX: All paths converge to global minimum', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Convex convergence
    ax = axes[0, 1]
    for x0, c in zip(starting_points, colors):
        x_hist, f_hist = gradient_descent_1d(f_convex, grad_convex, x0, lr=0.2)
        ax.semilogy(f_hist, 'o-', color=c, label=f'x0={x0}')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('f(x) (log scale)')
    ax.set_title('Convex: Convergence to zero', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Non-convex function plot
    y_nonconvex = f_nonconvex(x)

    ax = axes[1, 0]
    ax.plot(x, y_nonconvex, 'b-', linewidth=2, label='f(x) = x^4 - 2x^2 + 0.5x')

    # Run GD from different starting points
    starting_points_nc = [-2.0, 0.0, 2.0]

    for x0, c in zip(starting_points_nc, colors):
        x_hist, f_hist = gradient_descent_1d(f_nonconvex, grad_nonconvex, x0, lr=0.05)
        ax.plot(x_hist, f_hist, 'o-', color=c, markersize=6,
                label=f'Start x0={x0}', alpha=0.7)
        ax.scatter([x_hist[0]], [f_hist[0]], color=c, s=150, marker='s',
                   edgecolors='black', zorder=5)
        ax.scatter([x_hist[-1]], [f_hist[-1]], color=c, s=150, marker='*',
                   edgecolors='black', zorder=5)

    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.set_title('NON-CONVEX: Different starting points -> different minima!', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Non-convex convergence
    ax = axes[1, 1]
    for x0, c in zip(starting_points_nc, colors):
        x_hist, f_hist = gradient_descent_1d(f_nonconvex, grad_nonconvex, x0, lr=0.05)
        ax.plot(f_hist, 'o-', color=c, label=f'x0={x0}, final={f_hist[-1]:.3f}')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('f(x)')
    ax.set_title('Non-convex: Converge to different values', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

compare_convex_nonconvex()


## 4. Interactive Gradient Descent Explorer

Use the sliders below to experiment with:
- **Learning rate**: Too small = slow convergence, too large = divergence
- **Starting point**: Where optimization begins
- **Function**: Different functions have different landscapes


In [ ]:
def interactive_gd_explorer(lr=0.1, x0=2.0, func_choice='quadratic'):
    """Interactive gradient descent visualization."""

    functions = {
        'quadratic': (lambda x: x**2, lambda x: 2*x, 'f(x) = x^2'),
        'quartic': (lambda x: x**4, lambda x: 4*x**3, 'f(x) = x^4'),
        'abs': (lambda x: np.abs(x), lambda x: np.sign(x), 'f(x) = |x|'),
        'exp': (lambda x: np.exp(x) - x, lambda x: np.exp(x) - 1, 'f(x) = e^x - x'),
        'nonconvex': (lambda x: x**4 - 2*x**2, lambda x: 4*x**3 - 4*x, 'f(x) = x^4 - 2x^2'),
    }

    f, grad_f, name = functions[func_choice]

    # Run gradient descent
    x_hist, f_hist = gradient_descent_1d(f, grad_f, x0, lr=lr, max_iter=50)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot function and GD path
    ax = axes[0]
    x = np.linspace(-4, 4, 500)
    y = f(x)
    ax.plot(x, y, 'b-', linewidth=2, label=name)

    # Plot GD path
    ax.plot(x_hist, f_hist, 'ro-', markersize=8, alpha=0.7, label='GD path')
    ax.scatter([x_hist[0]], [f_hist[0]], color='green', s=200, marker='s',
               edgecolors='black', zorder=5, label=f'Start (x={x0})')
    ax.scatter([x_hist[-1]], [f_hist[-1]], color='red', s=200, marker='*',
               edgecolors='black', zorder=5, label=f'End (x={x_hist[-1]:.3f})')

    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('f(x)', fontsize=12)
    ax.set_title(f'{name}\nLR={lr}, Start={x0}, Final={x_hist[-1]:.4f}', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(min(y) - 1, min(max(y), 20))

    # Convergence plot
    ax = axes[1]
    ax.plot(f_hist, 'bo-', markersize=6)
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel('f(x)', fontsize=12)
    ax.set_title(f'Convergence ({len(x_hist)-1} iterations)', fontsize=12)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print summary
    print(f"\nResults:")
    print(f"  Starting point: x = {x0}")
    print(f"  Final point: x = {x_hist[-1]:.6f}")
    print(f"  Final value: f(x) = {f_hist[-1]:.6f}")
    print(f"  Iterations: {len(x_hist)-1}")

# Interactive version (if widgets available)
if WIDGETS_AVAILABLE:
    interact(interactive_gd_explorer,
             lr=FloatSlider(min=0.01, max=0.5, step=0.01, value=0.1, description='Learning Rate'),
             x0=FloatSlider(min=-3.0, max=3.0, step=0.1, value=2.0, description='Start x0'),
             func_choice=Dropdown(options=['quadratic', 'quartic', 'abs', 'exp', 'nonconvex'],
                                  value='quadratic', description='Function'))
else:
    # Static version
    print("Running static version (install ipywidgets for interactive sliders)")
    interactive_gd_explorer(lr=0.1, x0=2.0, func_choice='quadratic')
    interactive_gd_explorer(lr=0.3, x0=-2.5, func_choice='nonconvex')


## 5. KKT Optimality Conditions

For constrained optimization:
$$\min_x f(x) \quad \text{s.t.} \quad g_i(x) \leq 0, \quad h_j(x) = 0$$

The **Karush-Kuhn-Tucker (KKT)** conditions are:
1. **Stationarity**: $\nabla f(x^*) + \sum_i \mu_i \nabla g_i(x^*) + \sum_j \lambda_j \nabla h_j(x^*) = 0$
2. **Primal feasibility**: $g_i(x^*) \leq 0$, $h_j(x^*) = 0$
3. **Dual feasibility**: $\mu_i \geq 0$
4. **Complementary slackness**: $\mu_i g_i(x^*) = 0$


In [ ]:
def verify_kkt_conditions():
    """
    Example: Minimize f(x,y) = x^2 + y^2 subject to x + y >= 1

    Rewrite constraint as: g(x,y) = 1 - x - y <= 0

    Lagrangian: L(x,y,mu) = x^2 + y^2 + mu*(1 - x - y)

    KKT conditions:
    1. grad_x L = 2x - mu = 0 => x = mu/2
    2. grad_y L = 2y - mu = 0 => y = mu/2
    3. g(x,y) <= 0 => 1 - x - y <= 0
    4. mu >= 0
    5. mu * g(x,y) = 0
    """

    print("=" * 60)
    print("KKT CONDITIONS VERIFICATION")
    print("=" * 60)
    print("\nProblem: min x^2 + y^2  s.t.  x + y >= 1")
    print("\nRewritten: min x^2 + y^2  s.t.  g(x,y) = 1 - x - y <= 0")
    print("\nLagrangian: L(x,y,mu) = x^2 + y^2 + mu*(1 - x - y)")

    # Solve KKT analytically
    # From stationarity: x = y = mu/2
    # If constraint is active: x + y = 1 => mu/2 + mu/2 = 1 => mu = 1
    # So x* = y* = 0.5, mu* = 1

    x_star, y_star, mu_star = 0.5, 0.5, 1.0

    print("\n" + "-" * 40)
    print("SOLUTION: x* = 0.5, y* = 0.5, mu* = 1.0")
    print("-" * 40)

    # Verify each condition
    print("\n1. STATIONARITY:")
    grad_f_x = 2 * x_star
    grad_f_y = 2 * y_star
    grad_g_x = -1
    grad_g_y = -1
    stat_x = grad_f_x + mu_star * grad_g_x
    stat_y = grad_f_y + mu_star * grad_g_y
    print(f"   grad_x L = 2x - mu = 2({x_star}) - {mu_star} = {stat_x} {'= 0 [OK]' if abs(stat_x) < 1e-10 else '[FAIL]'}")
    print(f"   grad_y L = 2y - mu = 2({y_star}) - {mu_star} = {stat_y} {'= 0 [OK]' if abs(stat_y) < 1e-10 else '[FAIL]'}")

    print("\n2. PRIMAL FEASIBILITY:")
    g_val = 1 - x_star - y_star
    print(f"   g(x*,y*) = 1 - {x_star} - {y_star} = {g_val} {'<= 0 [OK]' if g_val <= 1e-10 else '[FAIL]'}")

    print("\n3. DUAL FEASIBILITY:")
    print(f"   mu* = {mu_star} {'>= 0 [OK]' if mu_star >= 0 else '[FAIL]'}")

    print("\n4. COMPLEMENTARY SLACKNESS:")
    comp_slack = mu_star * g_val
    print(f"   mu* * g(x*,y*) = {mu_star} * {g_val} = {comp_slack} {'= 0 [OK]' if abs(comp_slack) < 1e-10 else '[FAIL]'}")

    print("\n" + "=" * 60)
    print("ALL KKT CONDITIONS SATISFIED!")
    print("=" * 60)

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Contour plot
    ax = axes[0]
    x = np.linspace(-0.5, 2, 100)
    y = np.linspace(-0.5, 2, 100)
    X, Y = np.meshgrid(x, y)
    Z = X**2 + Y**2

    # Objective contours
    levels = [0.1, 0.25, 0.5, 1, 2, 3, 4]
    cs = ax.contour(X, Y, Z, levels=levels, cmap='Blues')
    ax.clabel(cs, inline=True, fontsize=8)

    # Constraint boundary: x + y = 1
    x_line = np.linspace(-0.5, 2, 100)
    y_line = 1 - x_line
    ax.plot(x_line, y_line, 'r-', linewidth=2, label='x + y = 1 (constraint boundary)')
    ax.fill_between(x_line, y_line, 2, alpha=0.2, color='green', label='Feasible region (x+y >= 1)')

    # Optimal point
    ax.scatter([x_star], [y_star], color='red', s=200, marker='*',
               edgecolors='black', zorder=5, label=f'Optimal: ({x_star}, {y_star})')

    # Gradient arrows at optimal point
    scale = 0.3
    ax.arrow(x_star, y_star, -grad_f_x*scale, -grad_f_y*scale, head_width=0.05,
             color='blue', linewidth=2, label='-grad f')
    ax.arrow(x_star, y_star, -grad_g_x*scale*mu_star, -grad_g_y*scale*mu_star,
             head_width=0.05, color='green', linewidth=2, label='-mu*grad g')

    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('KKT: Gradients Balance at Optimum', fontsize=12)
    ax.legend(loc='upper right', fontsize=8)
    ax.set_xlim(-0.5, 2)
    ax.set_ylim(-0.5, 2)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    # 3D surface
    ax = axes[1]
    ax = fig.add_subplot(122, projection='3d')

    # Mask infeasible region
    Z_masked = np.where(X + Y >= 1, Z, np.nan)

    surf = ax.plot_surface(X, Y, Z_masked, cmap='viridis', alpha=0.7,
                           linewidth=0, antialiased=True)

    # Optimal point
    ax.scatter([x_star], [y_star], [x_star**2 + y_star**2], color='red', s=200,
               marker='*', edgecolors='black', zorder=5)

    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('f(x,y)')
    ax.set_title('Objective Surface (Feasible Region)', fontsize=12)

    plt.tight_layout()
    plt.show()

verify_kkt_conditions()


In [ ]:
def interactive_kkt_explorer(constraint_rhs=1.0):
    """
    Explore how changing the constraint affects the KKT solution.

    Problem: min x^2 + y^2  s.t.  x + y >= b
    """
    b = constraint_rhs

    # Analytical solution
    # x = y = mu/2, x + y = b => mu = b
    # So x* = y* = b/2
    if b > 0:
        x_star = y_star = b / 2
        mu_star = b
    else:
        x_star = y_star = 0
        mu_star = 0

    f_star = x_star**2 + y_star**2

    # Visualization
    fig, ax = plt.subplots(figsize=(8, 8))

    x = np.linspace(-1, 3, 100)
    y = np.linspace(-1, 3, 100)
    X, Y = np.meshgrid(x, y)
    Z = X**2 + Y**2

    # Objective contours
    levels = [0.1, 0.25, 0.5, 1, 2, 3, 4, 6, 8]
    cs = ax.contour(X, Y, Z, levels=levels, cmap='Blues')
    ax.clabel(cs, inline=True, fontsize=8)

    # Constraint boundary
    x_line = np.linspace(-1, 3, 100)
    y_line = b - x_line
    ax.plot(x_line, y_line, 'r-', linewidth=2, label=f'x + y = {b}')
    ax.fill_between(x_line, y_line, 3, alpha=0.2, color='green')

    # Optimal point
    ax.scatter([x_star], [y_star], color='red', s=200, marker='*',
               edgecolors='black', zorder=5)
    ax.annotate(f'Optimal\n({x_star:.2f}, {y_star:.2f})\nf* = {f_star:.3f}',
                xy=(x_star, y_star), xytext=(x_star+0.5, y_star+0.5),
                fontsize=10, arrowprops=dict(arrowstyle='->', color='black'))

    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('y', fontsize=12)
    ax.set_title(f'KKT Solution for x + y >= {b}\nmu* = {mu_star:.2f}', fontsize=12)
    ax.set_xlim(-1, 3)
    ax.set_ylim(-1, 3)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"Constraint RHS (b): {b}")
    print(f"Optimal solution: x* = {x_star:.4f}, y* = {y_star:.4f}")
    print(f"Optimal value: f* = {f_star:.4f}")
    print(f"Lagrange multiplier: mu* = {mu_star:.4f}")
    print(f"\nInterpretation: mu* = {mu_star:.4f} means relaxing the constraint by 1 unit")
    print(f"would decrease the optimal value by approximately {mu_star:.4f}")

if WIDGETS_AVAILABLE:
    interact(interactive_kkt_explorer,
             constraint_rhs=FloatSlider(min=0.0, max=3.0, step=0.1, value=1.0,
                                        description='Constraint b'))
else:
    for b in [0.5, 1.0, 2.0]:
        interactive_kkt_explorer(b)


---

## Interview Questions

### Question 1: Convexity and Optimization
**Q: Why is convexity important in optimization? What guarantees do we have for convex problems that we don't have for non-convex problems?**

<details>
<summary>Click for Answer</summary>

**Key Points:**
1. **Global Optimality**: For convex problems, any local minimum is also a global minimum. This means gradient descent will find the best solution, not just a "pretty good" one.

2. **Efficient Algorithms**: Convex problems can be solved efficiently (polynomial time) with well-understood algorithms.

3. **Unique Solution**: Strictly convex problems have at most one global minimum.

4. **Convergence Guarantees**: For smooth convex functions, gradient descent converges at rate O(1/k) or O(1/k^2) with acceleration.

5. **KKT Sufficiency**: For convex problems with convex constraints, KKT conditions are both necessary AND sufficient for optimality.

**Non-convex problems** may have:
- Multiple local minima
- Saddle points
- No polynomial-time algorithm guaranteed to find global optimum
- NP-hard in general
</details>

### Question 2: KKT Conditions
**Q: Explain the complementary slackness condition in KKT. What is its intuitive meaning?**

<details>
<summary>Click for Answer</summary>

**Complementary Slackness**: $\mu_i g_i(x^*) = 0$ for each inequality constraint.

**Intuition:**
- Either the constraint is **inactive** ($g_i(x^*) < 0$) and its multiplier is zero ($\mu_i = 0$)
- Or the constraint is **active** ($g_i(x^*) = 0$) and may have non-zero multiplier ($\mu_i \geq 0$)

**Physical Interpretation:**
- $\mu_i$ represents the "price" of the constraint
- If a constraint doesn't bind (slack), it doesn't affect the solution, so its price is zero
- If a constraint binds, it's "pushing" against the solution, and $\mu_i$ measures how much the objective would improve if we relaxed that constraint

**Example:** In resource allocation, if you have unused capacity ($g_i < 0$), adding more of that resource doesn't help (so its value $\mu_i = 0$). But if a resource is fully used ($g_i = 0$), more of it would be valuable ($\mu_i > 0$).
</details>

### Question 3: Learning Rate Selection
**Q: In gradient descent, what happens if the learning rate is too large or too small? How would you choose an appropriate learning rate in practice?**

<details>
<summary>Click for Answer</summary>

**Too Small Learning Rate:**
- Very slow convergence
- May take impractically long to reach minimum
- Gets stuck in flat regions
- Wastes computational resources

**Too Large Learning Rate:**
- Oscillations around minimum
- May diverge (function value increases)
- Overshoots the minimum
- Unstable optimization

**Practical Selection Methods:**

1. **Line Search**: At each step, search for optimal step size satisfying Wolfe conditions

2. **Learning Rate Schedules**:
   - Start large, decay over time
   - Step decay: reduce by factor every N epochs
   - Exponential decay: $\alpha_t = \alpha_0 e^{-kt}$
   - Cosine annealing

3. **Adaptive Methods**:
   - Adam: maintains per-parameter learning rates
   - AdaGrad: adapts based on historical gradients
   - RMSprop: uses moving average of squared gradients

4. **Theory-Guided**:
   - For L-smooth convex: use $\alpha \leq 1/L$
   - For strongly convex: use $\alpha \leq 2/(\mu + L)$

5. **Grid Search**: Try powers of 10 (0.001, 0.01, 0.1, 1.0)
</details>

---


## Exercises

### Exercise 1: Convexity Verification
Implement a function to check convexity of a 2D function by sampling. Test it on:
- $f(x,y) = x^2 + y^2$ (convex)
- $f(x,y) = xy$ (not convex)
- $f(x,y) = \max(x, y)$ (convex)


In [ ]:
# Exercise 1: Your solution here
def check_convexity_2d(f, x_range=(-5, 5), y_range=(-5, 5), n_tests=1000, tol=1e-6):
    """
    Check if a 2D function is convex by sampling.

    TODO: Implement this function

    Hint: Sample random points (x1, y1) and (x2, y2), then check if
    f(lam*(x1,y1) + (1-lam)*(x2,y2)) <= lam*f(x1,y1) + (1-lam)*f(x2,y2)
    """
    pass

# Test your implementation
# f1 = lambda x, y: x**2 + y**2
# print(f"x^2 + y^2 is convex: {check_convexity_2d(f1)}")


### Exercise 2: Gradient Descent with Momentum
Implement gradient descent with momentum and compare convergence with vanilla GD:
$$v_{t+1} = \beta v_t + \nabla f(x_t)$$
$$x_{t+1} = x_t - \alpha v_{t+1}$$


In [ ]:
# Exercise 2: Your solution here
def gradient_descent_momentum(f, grad_f, x0, lr=0.1, beta=0.9, max_iter=100):
    """
    Gradient descent with momentum.

    TODO: Implement this function

    Parameters:
    -----------
    beta : float
        Momentum coefficient (typically 0.9)

    Returns:
    --------
    x_history, f_history
    """
    pass

# Test your implementation
# Compare GD vs GD with momentum on f(x) = x^2


### Exercise 3: Constrained Optimization
Solve the following problem using the KKT conditions:

$$\min_{x,y} \quad x + 2y$$
$$\text{s.t.} \quad x^2 + y^2 \leq 1$$

1. Write out the Lagrangian
2. Derive the KKT conditions
3. Find the optimal solution
4. Verify all KKT conditions are satisfied


In [ ]:
# Exercise 3: Your solution here

# 1. Lagrangian: L(x,y,mu) = x + 2y + mu*(x^2 + y^2 - 1)

# 2. KKT conditions:
#    Stationarity: 1 + 2*mu*x = 0 => x = -1/(2*mu)
#                  2 + 2*mu*y = 0 => y = -1/mu
#    Primal feasibility: x^2 + y^2 <= 1
#    Dual feasibility: mu >= 0
#    Complementary slackness: mu*(x^2 + y^2 - 1) = 0

# 3. Solve for optimal solution
# TODO: Find x*, y*, mu*

# 4. Verify KKT conditions
# TODO: Check all conditions are satisfied
